In [ ]:
%%shell
# Ubuntu no longer distributes chromium-browser outside of snap
#
# Proposed solution: https://askubuntu.com/questions/1204571/how-to-install-chromium-without-snap

# Add debian buster
cat > /etc/apt/sources.list.d/debian.list <<'EOF'
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster.gpg] http://deb.debian.org/debian buster main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster-updates.gpg] http://deb.debian.org/debian buster-updates main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-security-buster.gpg] http://deb.debian.org/debian-security buster/updates main
EOF

# Add keys
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A

apt-key export 77E11517 | gpg --dearmour -o /usr/share/keyrings/debian-buster.gpg
apt-key export 22F3D138 | gpg --dearmour -o /usr/share/keyrings/debian-buster-updates.gpg
apt-key export E562B32A | gpg --dearmour -o /usr/share/keyrings/debian-security-buster.gpg

# Prefer debian repo for chromium* packages only
# Note the double-blank lines between entries
cat > /etc/apt/preferences.d/chromium.pref << 'EOF'
Package: *
Pin: release a=eoan
Pin-Priority: 500


Package: *
Pin: origin "deb.debian.org"
Pin-Priority: 300


Package: chromium*
Pin: origin "deb.debian.org"
Pin-Priority: 700
EOF

# Install chromium and chromium-driver
apt-get update
apt-get install chromium chromium-driver

# Install selenium
pip install selenium

Executing: /tmp/apt-key-gpghome.7dhAHXKsPd/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
gpg: key DCC9EFBF77E11517: public key "Debian Stable Release Key (10/buster) <debian-release@lists.debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.Jurej93VdD/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
gpg: key DC30D7C23CBBABEE: public key "Debian Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.1zj8RCHbXi/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A
gpg: key 4DFAB270CAA96DFA: public key "Debian Security Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Get:1 http://deb.debian.org/debian buster InRelease [122 kB]
Get:2 http://deb.debian.org/debian bust

In [ ]:
#!pip install selenium==4.1.0
!pip install beautifulsoup4
!pip install lxml
!pip install pandas
!pip install webdriver-manager
!pip install python-dateutil
!pip install bottle
!pip install pivottablejs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 2.6 MB/s eta 0:00:00


In [ ]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime

from bs4 import BeautifulSoup
from dateutil.relativedelta import relativedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import pandas
import requests
import re
#import constants

current_year = datetime.now().year

#to parse datetime into format
def parse_date(date: str) -> str:
    date_format = "%A, %B %d, at %I:%M %p"
    datetime_obj = datetime.strptime(date, date_format)
    day = datetime_obj.strftime("%d")
    month = datetime_obj.strftime("%m")
    time = datetime_obj.strftime("%H:%M")
    parsed_date = f"{current_year}-{month}-{day} {time}:00"
    return parsed_date

#to check if event ends next year
def event_ends_next_year(start_date: str, end_date: str):
    start_month = start_date[5:7]
    end_month = end_date[5:7]
    return int(start_month) == 12 and int(end_month) < 12

#to check if event is all day
def is_all_day_event(start_date: str, end_date: str):
    start_month_and_day = start_date[5:10]
    end_month_and_day = end_date[5:10]
    start_time = start_date[11:]
    end_time = end_date[11:]

    return (
        start_month_and_day == end_month_and_day
        and start_time == "00:00:00"
        and end_time == "23:59:00"
    )

#convert date to RFC 3339
def convert_to_rfc3339(date: str):
    rfc3339_format = "%Y-%m-%dT%H:%M:%S"
    date_object = datetime.strptime(date, "%Y-%m-%d %H:%M:%S")

    return date_object.strftime(rfc3339_format)

#convert to year-month-date
def convert_to_yyy_mm_dd(date: str):
    yyy_mm_dd_format = "%Y-%m-%d"
    date_object = datetime.strptime(date, "%Y-%m-%d %H:%M:%S")
    return date_object.strftime(yyy_mm_dd_format)

#get data from url passed
def getdata(url):
    r = requests.get(url)
    return r.text

#Class to parse the structure required for output
@dataclass
class Event:
    start_time: str = field(compare=False)
    end_time: str = field(compare=False)
    summary: str
    description: str
    img_src : str
    bonus:list
    timeZone:str
    Description:str
    pokemonId:list

    def to_dict(self):
        if is_all_day_event(self.start_time, self.end_time):
            self.start_time = convert_to_yyy_mm_dd(self.start_time)
            self.end_time = convert_to_yyy_mm_dd(self.end_time)

            metadata = {
                "summary": self.summary,
                "description": self.description,
                "start": {"date": self.start_time},
                "end": {"date": self.end_time},
                "img_src": self.img_src,
                "pokemonId" : self.pokemonId,
                "Bonus": self.bonus,
                "timeZone": self.timeZone,
                "Description":self.Description,
            }

        elif event_ends_next_year(self.start_time, self.end_time):
            self.start_time = convert_to_rfc3339(self.start_time)

            end_time_date_object = datetime.strptime(self.end_time, "%Y-%m-%d %H:%M:%S")
            end_time_date_object = end_time_date_object + relativedelta(year=1)

            self.end_time = end_time_date_object.strftime("%Y-%m-%d %H:%M:%S")
            self.end_time = convert_to_rfc3339(self.end_time)

            metadata = {
                "summary": self.summary,
                "description": self.description,
                "start": {"dateTime": self.start_time},
                "end": {"dateTime": self.end_time},
                "img_src": self.img_src,
                "pokemonId" : self.pokemonId,
                "Bonus": self.bonus,
                "timeZone": self.timeZone,
                "Description":self.Description,
            }

        else:
            metadata = {
                "summary": self.summary,
                "description": self.description,
                "start": {"dateTime": self.start_time},
                "end": {"dateTime": self.end_time},
                "img_src": self.img_src,
                "pokemonId" : self.pokemonId,
                "Bonus": self.bonus,
                "timeZone": self.timeZone,
                "Description":self.Description,
            }

        return metadata

    def __str__(self):
        return str(self.to_dict())

#Main Method
def main():
    events = defaultdict()
    service = Service(executable_path=r'/usr/bin/chromedriver')
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")

    options.headless = True

    driver = webdriver.Chrome(service=service, options=options)
    #driver = webdriver.Chrome(ChromeDriverManager().install())
    #URL of leekducks
    url = "https://leekduck.com/events"

    driver.get(url)

    #get the lxml of the url page
    soup = BeautifulSoup(driver.page_source, "lxml")

    #find all div with class name "current-events"
    soup = soup.find_all("div", class_="current-events")[0]

    #find all spans with class="event-header-item-wrapper"
    soup = soup.find_all(
        "span",
        class_="event-header-item-wrapper"
    )
    timeZone = str()
    Description = str()
    event_links = set()
    img_src = list()

    for span in soup:
        #get all href from a in html of the url
        event_name = span.find("a").get("href")
        #if event is uannounced then pass since we dont have any deatils on the event
        if "unannounced" in event_name:
            continue
        link = f"https://leekduck.com{event_name}"
        #append list with url for all events in leekduck page
        event_links.add(link)

    #iterate through each link to obtain date
    for link in event_links:
        soup_bonus = []
        htmldata = getdata(link)
        soup = BeautifulSoup(htmldata, 'html.parser')
        soup2 = BeautifulSoup(driver.page_source, "lxml")

        img_src = []
        pokemonId = []
        #find all "img" in the event page
        for item in soup.find_all('img'):
            #get the source of the image so that i can used in the front end
            src = item.get("src")
            if src:
                src = requests.compat.urljoin(url, src)
                #to always include the first link in the img src list
                if len(img_src)==0 or 'pokemon_icons' in src:
                    if '_crop' not in src:
                        if src not in img_src:
            #img_link='https://leekduck.com'+item['src']
                            img_src.append(src)
                            #find a digits in the img src string to get the pokemon id
                            pokemonIdData = re. findall('\d+', src)
                            if len(pokemonIdData)>0:
                                # if the digits is less or equal to 3 and pokemon_icons is in string of img src:
                                if len(pokemonIdData[0])<=3 and 'pokemon_icons' in src:
                                    #remove leading zeroes from pokemon id , since it is to be used in pokeapi api
                                    pokemonIdCleaned = pokemonIdData[0].lstrip("0")
                                    #check for mega pokemon
                                    if '_51.png' in src:
                                        pokemonId.append(str(pokemonIdCleaned+'_51'))
                                    elif '_52.png' in src:
                                        pokemonId.append(str(pokemonIdCleaned+'_52'))
                                    elif 'fMEGA' in src:
                                        pokemonId.append(str(pokemonIdCleaned+'fMEGA'))
                                    elif '_61.png' in src:
                                        pokemonId.append(str(pokemonIdCleaned+'_61'))
                                    elif 'fHISUIAN' in src:
                                        pokemonId.append(str(pokemonIdCleaned+'fHISUIAN'))
                                    else:
                                        pokemonId.append(pokemonIdCleaned)
                                        #print(pokemonIdData[0].lstrip("0"))
                elif 'shiny-icon' in src:
                    #prev_img_link=soup.findall("li",class_='pkmn-list-item')
                    img_poke_shiny=img_src[len(img_src)-1]
                    if '.icon.' in img_poke_shiny and (img_poke_shiny.count('.s')<2):
                        #print(img_poke_shiny)
                        #print(img_poke_shiny.count('.s'))
                        img_poke_shiny=img_poke_shiny.split('.icon.')
                        img_poke_shiny=img_poke_shiny[0]+'.s.icon.png'
                        if img_poke_shiny not in img_src and img_poke_shiny.count('.s')<2:
                            img_src.append(img_poke_shiny)
                            pokemonIdData = re. findall('\d+', img_poke_shiny)
                            if len(pokemonIdData)>0:
                                # if the digits is less or equal to 3 and pokemon_icons is in string of img src:
                                if len(pokemonIdData[0])<=3 and 'pokemon_icons' in img_poke_shiny:
                                    #remove leading zeroes from pokemon id , since it is to be used in pokeapi api
                                    pokemonIdCleaned = pokemonIdData[0].lstrip("0")
                                    #check for mega pokemon
                                    if '_51.png' in img_poke_shiny or '_51.s.icon.png' in img_poke_shiny:
                                        #print(img_poke_shiny)
                                        pokemonId.append(str(pokemonIdCleaned+'_51'))
                                    elif '_52.png' in img_poke_shiny or '_52.s.icon.png' in img_poke_shiny :
                                        pokemonId.append(str(pokemonIdCleaned+'_52'))
                                    elif 'fMEGA' in img_poke_shiny:
                                        pokemonId.append(str(pokemonIdCleaned+'fMEGA'))
                                    elif '_61.png' in img_poke_shiny  or '_61.s.icon.png' in img_poke_shiny :
                                        pokemonId.append(str(pokemonIdCleaned+'_61'))
                                    elif 'fHISUIAN' in img_poke_shiny:
                                        pokemonId.append(str(pokemonIdCleaned+'fHISUIAN'))
                                    else:
                                        pokemonId.append(pokemonIdCleaned)

                    elif '_shiny' in img_poke_shiny and  (img_poke_shiny.count('_shiny')<2 and img_poke_shiny.count('.s')<2):

                        img_poke_shiny=img_poke_shiny.split('.png')
                        img_poke_shiny=img_poke_shiny[0]+'_shiny.png'
                        if img_poke_shiny not in img_src and img_poke_shiny.count('_shiny')<2 and img_poke_shiny.count('.s')<2 :
                            img_src.append(img_poke_shiny)
                            pokemonIdData = re. findall('\d+', img_poke_shiny)
                            if len(pokemonIdData)>0:
                                # if the digits is less or equal to 3 and pokemon_icons is in string of img src:
                                if len(pokemonIdData[0])<=3 and 'pokemon_icons' in img_poke_shiny:
                                    #remove leading zeroes from pokemon id , since it is to be used in pokeapi api
                                    pokemonIdCleaned = pokemonIdData[0].lstrip("0")
                                    #check for mega pokemon
                                    if '_51.png' in img_poke_shiny or '_51_shiny.png' in img_poke_shiny:
                                        pokemonId.append(str(pokemonIdCleaned+'_51'))
                                    elif '_52.png' in img_poke_shiny or '_52_shiny.png' in img_poke_shiny:
                                        pokemonId.append(str(pokemonIdCleaned+'_52'))
                                    elif 'fMEGA' in img_poke_shiny:
                                        pokemonId.append(str(pokemonIdCleaned+'fMEGA'))
                                    elif '_61.png' in img_poke_shiny or '_61_shiny.png' in img_poke_shiny:
                                        pokemonId.append(str(pokemonIdCleaned+'_61'))
                                    elif 'fHISUIAN' in img_poke_shiny:
                                        pokemonId.append(str(pokemonIdCleaned+'fHISUIAN'))
                                    else:
                                        pokemonId.append(pokemonIdCleaned)
        #img_src.pop()
        #obtain all div with class="bonus" to get bonus details for thet event
        for soups in soup.find_all("div",class_="bonus-text"):
            soup_bonus.append(soups.string)
        #get all span with event end time for that event to get the timezone
        for span in soup2.find_all('span', {'id': 'event-time-end'}):
            timeZoneString = span.string
            #cleaning of data
            timeZoneSplit = timeZoneString.split('M')
            if len(timeZoneSplit) == 1:
                timeZone = 'Local Time'
            else:
              timeZone = timeZoneSplit[1]
        driver.get(link)
        soup = BeautifulSoup(driver.page_source, "lxml")

        #find the description of the event
        for p in soup.find_all("div",class_="event-description"):
            #find the paragraph to get event description
            data = p.find_all('p')
            #cleaning data
            desc = str(data[0]).replace('<p>','')
            Description = desc.replace('</p>','')

        #get the title of the event
        title = soup.find("h1").text.strip()

        #get start date of the event
        start_date = (
            WebDriverWait(driver, 10)
            .until(EC.presence_of_element_located((By.ID, "event-date-start")))
            .text.strip()
            .rstrip(",")
            .replace("  ", " ")
        )
        #get start time of the event
        start_time = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "event-time-start"))
        ).text.split("M")[0] + "M".replace("  ", " ")

        #calculate start date time
        complete_start_date = f"{start_date}, {start_time}"

        #get end date
        end_date = (
            WebDriverWait(driver, 10)
            .until(EC.presence_of_element_located((By.ID, "event-date-end")))
            .text.strip()
            .rstrip(",")
            .replace("  ", " ")
        )
        #get end time
        end_time = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "event-time-end"))
        ).text.split("M")[0] + "M".replace("  ", " ")

        #calculate end date time
        complete_end_date = f"{end_date}, {end_time}"

        #if start date not empty , parse the dates and pass data to Event class to structure it
        if start_date != "None":
            parsed_start_date = parse_date(complete_start_date)
            parsed_end_date = parse_date(complete_end_date)

            new_event = Event(parsed_start_date, parsed_end_date, title, link,img_src,soup_bonus,timeZone,Description,pokemonId)
            events[link] = new_event

    #create a dataframe of events to be saved as csv
    df = pandas.DataFrame(events,index=[0]).T
    df.columns = ['Summary']
    #df.to_csv(constants.URL+'file.csv')
    df.to_csv('./file.csv')
    driver.quit()


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import json
from bottle import *
from pivottablejs import pivot_ui
import re
from ast import literal_eval
import requests
from urllib.request import Request,urlopen
import shutil
import time
from urllib.request import urlopen
import base64



links=[]
summaries=[]
jsonSummary=[]
dfSummaries=[]
startDate=[]
startDateV2=[]
imgSrc=[]
bonus=[]
timeZone=[]
endDate=[]
description=[]
data=[]
pokemonId=[]
dateDuration=[]
pokemonType=[]
preference=[]
encodedData64 = []
encodedData64Data = []

df=pd.read_csv('./file.csv')

pokemonData=pd.DataFrame()

def getPokeName(pokeId,mainurl,req):
    start_time=time.time()
    responseJSON=urlopen(req)
    #print(time.time()-start_time)
    dataJSONType=json.loads(responseJSON.read())
    typeData=dataJSONType['forms']
    typeData=str(typeData).split('name')
    typeData=typeData[1].split(',')
    typeData=typeData[0].split(':')
    typeData=typeData[1].replace(" ","")
    typeData=typeData.replace('\'','')
    return typeData

def poke_51_52_61_fMEGA(pokeId,id,url,typeData):

    if id == '_51':
        if pokeId == '6' or pokeId =='150':
            newurl=url+(typeData+'-mega-x')
        else:
            newurl=url+(typeData+'-mega')
    elif id == '_52':
        newurl=url+(typeData+'-mega-y')
    elif id == 'fMEGA':
        newurl=url+(typeData+'-mega')
    elif id == '_61':
        newurl=url+(typeData+'-alola')
    elif id == 'fHISUIAN':
        newurl = url+(typeData+'-hisui')
    else:
        if pokeId == '263':
          newurl = url+typeData
        else:
          newurl = url+pokeId

    return newurl

def poke_special(pokemonMiniType2,pokeId,url,id):
    if id == '_51':
        pokeId=pokeId.replace('_51','')
    elif id == '_52':
        pokeId=pokeId.replace('_52','')
    elif id == 'fMEGA':
        pokeId = pokeId.replace('fMEGA','')
    elif id == '_61':
        pokeId = pokeId.replace('_61','')
    elif id == 'fHISUIAN':
        pokeId = pokeId.replace('fHISUIAN','')
    mainurl=url+pokeId
    req = Request(
            url=mainurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    typeData=getPokeName(pokeId,mainurl,req)
    newurl=poke_51_52_61_fMEGA(pokeId,id,url,typeData)
    req2=Request(
            url=newurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    responseJSON2=urlopen(req2)
    dataJSONType2=json.loads(responseJSON2.read())
    for k in range(len(dataJSONType2['types'])):
        typeData2=dataJSONType2['types'][k]
        typeData2=str(typeData2).split('name')
        typeData2=typeData2[1].split(',')
        typeData2=typeData2[0].split(':')
        typeData2=typeData2[1].replace(" ","")
        typeData2=typeData2.replace('\'','')
        pokemonMiniType2.append(typeData2)
    return pokemonMiniType2

def calculate_preference(dateDuration):
    for i in dateDuration:
        preference.append(len(i))

for i in range(len(df)):
    links.append(df.iloc[i,0])
    summaries.append(df.iloc[i,1])

pokemonData['Links'] = links

for summary in summaries:
    summ=summary.split(',')
    i=2
    if 'start' not in summ[2]:
        i=3
        while 'start' not in summ[i]:
            i+=1
    summCleanedDate=summ[i].replace('{','')
    startDate.append(summCleanedDate.replace('}',''))

    jsonSummary.append(summ[0].replace('{',''))

for summary in summaries:
    summ=summary.split('img_src')
    summ=summ[1].split('Bonus')
    summ=summ[0].split(':',1)
    summ=summ[1].split('pokemonId')
    strToBeCleaned=summ[0]
    strToReplace='\''
    replacementStr=''
    pos=strToBeCleaned.rfind(strToReplace)
    if pos >-1:
        summCleanedSplit=strToBeCleaned[:pos]+replacementStr + strToBeCleaned[pos + len(strToReplace): ]
    else:
        summCleanedSplit=strToBeCleaned
    strToReplace2=','
    pos2=summCleanedSplit.rfind(strToReplace2)
    if pos2 >-1:
        summCleanedSplit=summCleanedSplit[:pos2]+replacementStr + summCleanedSplit[pos2 + len(strToReplace): ]
    else:
        summCleanedSplit=summCleanedSplit
    imgSrc.append(summCleanedSplit)

    """for link in list(summCleanedSplit.split(',')):
      linkCleaned = link.replace('[','').replace('\'','').replace(']','')
      request_site = Request(linkCleaned, headers={"User-Agent": "Mozilla/5.0"})
      encodedData = base64.b64encode(urlopen(request_site).read())
      encodedData64.append(encodedData)
    encodedData64Data.append(encodedData64)
    print(encodedData64Data)"""

for summary in summaries:
     summ= summary.split('timeZone')
     #summSplit=summ[1].split(':',1)
     summ=summ[0].rsplit('Bonus',1)
     summ=summ[1].split(':',1)
     summ=summ[1]
     #summCleaned=summSplit[1].replace('}','')
     #summCleanedSplit=summCleaned.split('timeZone')
     strToBeCleaned=summ
     strToReplace='\''
     replacementStr=''
     pos=strToBeCleaned.rfind(strToReplace)
     if pos >-1:
         summCleanedSplit=strToBeCleaned[:pos]+replacementStr + strToBeCleaned[pos + len(strToReplace): ]
     else:
         summCleanedSplit=strToBeCleaned
     strToReplace2=','
     pos2=summCleanedSplit.rfind(strToReplace2)
     if pos2>-1:
         summCleanedSplit=summCleanedSplit[:pos2]+replacementStr + summCleanedSplit[pos2 + len(strToReplace2): ]
     else:
        summCleanedSplit=strToBeCleaned
     bonus.append(summCleanedSplit)
     #print(summCleanedSplit)


for summary in summaries:
    summ=summary.split('end')
    summ=summ[1].split('img_src')
    summ=summ[0].split(':',1)
    summ=summ[1].split(':',1)
    summ=summ[1].replace('}','')
    summ=summ.replace(',','')
    strToReplace='\''
    replacementStr=''
    pos=summ.rfind(strToReplace)
    if pos >-1:
        summCleanedSplit=summ[:pos]+replacementStr + summ[pos + len(strToReplace): ]
    else:
        summCleanedSplit=summ
    summCleanedSplit=summCleanedSplit.replace('\'','')
    summCleanedSplit=summCleanedSplit.lstrip()
    summCleanedSplit=summCleanedSplit.rstrip()
    endDate.append(summCleanedSplit)

for summary in summaries:
    summ=summary.split('timeZone')
    summTimeZone=summ[1].split(':')
    summTimeZone=summTimeZone[1].split(',')
    summTimeZone=summTimeZone[0].replace('}','')
    summTimeZone=summTimeZone.replace('\'','')
    summTimeZone=summTimeZone.replace(' ','')
    if summTimeZone=="":
        summTimeZone="LocalTime"
        timeZone.append(summTimeZone.replace('\\xa0',''))
    elif summTimeZone!="":
        timeZone.append(summTimeZone.replace('\\xa0',''))

for summary in summaries:
    summ=summary.split('Description')
    summ=summ[1].split(':',1)
    summ=str(summ[1])
    summ=re.sub(r'<.*?>','',summ)
    summ=summ.replace('\'','')
    description.append(summ.replace('}','').lstrip())

for summary in summaries:
    summ=summary.split('pokemonId')
    summ=summ[1].split('Bonus')
    strToBeCleaned=summ[0]
    strToReplace=','
    replacementStr=''
    pos=strToBeCleaned.rfind(strToReplace)
    if pos >-1:
        summCleanedSplit=strToBeCleaned[:pos]+replacementStr + strToBeCleaned[pos + len(strToReplace): ]
    else:
        summCleanedSplit=strToBeCleaned
    summ=summCleanedSplit.split(':',1)
    strToBeCleaned2=summ[1]
    strToReplace2='\''
    replacementStr2=''
    pos2=strToBeCleaned2.rfind(strToReplace2)
    if pos2 >-1:
        summCleanedSplit=strToBeCleaned2[:pos2]+replacementStr2 + strToBeCleaned2[pos2 + len(strToReplace2): ]
    else:
        summCleanedSplit=strToBeCleaned2
    pokemonId.append(summCleanedSplit)
    pokemonIdSplit=summCleanedSplit.split(',')
    pokemonMiniType=[]
    url='https://pokeapi.co/api/v2/pokemon/'
    for i in pokemonIdSplit:
        pokemonMiniType2=[]
        pokeId=i.replace('\'','')
        pokeId=pokeId.replace('[','')
        pokeId=pokeId.replace(']','')
        pokeId=pokeId.lstrip()
        #pokeId=int(pokeId.strip() or 0)
        if len(pokeId)>0:
            if '_51' in pokeId:
                id = '_51'
                pokemonMiniType2 = poke_special(pokemonMiniType2,pokeId,url,id)
                pokemonMiniType.append(pokemonMiniType2)
            elif '_52' in pokeId:
                id = '_52'
                pokemonMiniType2 = poke_special(pokemonMiniType2,pokeId,url,id)
                pokemonMiniType.append(pokemonMiniType2)
            elif '_61' in pokeId:
                id = '_61'
                pokemonMiniType2 = poke_special(pokemonMiniType2,pokeId,url,id)
                pokemonMiniType.append(pokemonMiniType2)
            elif 'fMEGA' in pokeId:
                id = 'fMEGA'
                pokemonMiniType2 = poke_special(pokemonMiniType2,pokeId,url,id)
                pokemonMiniType.append(pokemonMiniType2)
            elif 'fHISUIAN' in pokeId:
                id = 'fHISUIAN'
                pokemonMiniType2 = poke_special(pokemonMiniType2,pokeId,url,id)
                pokemonMiniType.append(pokemonMiniType2)
            else:
                id=''
                pokemonMiniType2=poke_special(pokemonMiniType2,pokeId,url,id)
                pokemonMiniType.append(pokemonMiniType2)
        else:
            pokemonMiniType.append(list())
    pokemonType.append(pokemonMiniType)

for summary in jsonSummary:
    splitSum=summary.split(':')
    splitSumClean=splitSum[1].replace('\\xa0',' ')
    splitSumClean=splitSumClean.replace('\"','')
    splitSumClean=splitSumClean.replace('\'','')
    splitSumClean=re.sub(r'[^\x00-\x7F]+','',splitSumClean)
    dfSummaries.append(splitSumClean.lstrip())

for sdate in startDate:
    splitDate=sdate.split(':',1)
    #splitDate_v2=str(splitDate).split(':',1)
    splitDate=splitDate[1].split(':',1)
    splitDateString=splitDate[1].replace('\'','')
    splitDateString=splitDateString.lstrip()
    startDateV2.append(splitDateString)

for i in range(len(startDateV2)):

    startDateDuration=startDateV2[i].lstrip(' ').split(' ',1)
    endDateDuration=endDate[i].lstrip(' ').split(' ',1)
    try:
      duration=pd.date_range(start=startDateDuration[0],end=endDateDuration[0])
    except:
      duration = list(startDateDuration[0])

    gapDuration=[]
    for i in duration:
        try:
          match_str = re.search(r'\d{4}-\d{2}-\d{2}',str(i))
          res = datetime.strptime(match_str.group(), '%Y-%m-%d').date()

          gapDuration.append(str(res))
        except:
          res = startDateDuration[0]
          gapDuration.append(str(res))
    dateDuration.append(gapDuration)

calculate_preference(dateDuration)

pokemonData['Summary']=dfSummaries
pokemonData['Start DateTime'] = startDateV2
pokemonData['End DateTime'] = endDate
pokemonData['Duration'] = dateDuration
pokemonData['preference'] = preference
pokemonData['Img Src'] = imgSrc
pokemonData['pokemonId'] = pokemonId
pokemonData['type'] = pokemonType
pokemonData['Bonus'] = bonus
pokemonData['timeZone'] = timeZone
pokemonData['Description'] = description
#pokemonData['EncodedImages'] = encodedData64Data

pokemonData['json'] = pokemonData.to_json(orient='records', lines=True).splitlines()


for i in pokemonData['json']:
    data.append(json.loads(i))

for i in data:
    i['Img Src'] = literal_eval(i['Img Src'])
    i['Bonus'] = literal_eval(i['Bonus'])
    i['pokemonId'] = literal_eval(i['pokemonId'])

for j in data:
    for k in range(len(j['pokemonId'])):
        if '_51' in j['pokemonId'][k]:
            j['pokemonId'][k]=j['pokemonId'][k].replace('_51','')
        elif '_52' in j['pokemonId'][k]:
            j['pokemonId'][k]=j['pokemonId'][k].replace('_52','')
        elif 'fMEGA' in  j['pokemonId'][k]:
            j['pokemonId'][k]=j['pokemonId'][k].replace('fMEGA','')
        elif '_61' in j['pokemonId'][k]:
            j['pokemonId'][k]=j['pokemonId'][k].replace('_61','')
        elif 'fHISUIAN' in j['pokemonId'][k]:
            j['pokemonId'][k]=j['pokemonId'][k].replace('fHISUIAN','')

"""for k in data:
    for i in k['Img Src']:
        if '_shiny' in i:
            imageSrc=i.split('_shiny')
            imageSrc=imageSrc[0]+'.png'
            pokemonIdData = re. findall('\d+', imageSrc)
            pokemonIdCleaned = pokemonIdData[0].lstrip("0")
            if imageSrc not in k['Img Src']:
                k['Img Src'].append(imageSrc)
                k['pokemonId'].append(pokemonIdCleaned)
                print(imageSrc)
                if '_51.png' in imageSrc:
                    pokemonIdString=str(pokemonIdCleaned)
                    #add code to add types for pokemon
                    pokeURL='https://pokeapi.co/api/v2/pokemon/'
                    pokemonMiniType5=[]
                    pokemonMiniType4=poke_special(pokemonMiniType5,pokemonIdString,pokeURL,'_51')
                    #print(pokemonMiniType4)
                    k['type'].append(pokemonMiniType4)
                elif '_52.png' in imageSrc:
                    pokemonIdString=str(pokemonIdCleaned)
                    pokeURL='https://pokeapi.co/api/v2/pokemon/'
                    pokemonMiniType5=[]
                    pokemonMiniType4=poke_special(pokemonMiniType5,pokemonIdString,pokeURL,'_52')
                    #print(pokemonMiniType4)
                    k['type'].append(pokemonMiniType4)
                else:
                    pokemonIdString=str(pokemonIdCleaned)
                    pokeURL='https://pokeapi.co/api/v2/pokemon/'
                    pokemonMiniType5=[]
                    pokemonMiniType4=poke_special(pokemonMiniType5,pokemonIdString,pokeURL,' ')
                    #print(pokemonMiniType4)
                    k['type'].append(pokemonMiniType4)


        elif '.s.' in i:
            imageSrc=i.split('.s')
            imageSrc=imageSrc[0]+'.icon.png'
            pokemonIdData = re. findall('\d+', imageSrc)
            pokemonIdCleaned = pokemonIdData[0].lstrip("0")
            if imageSrc not in k['Img Src']:
                k['Img Src'].append(imageSrc)
                k['pokemonId'].append(pokemonIdCleaned)
                print(imageSrc)
                if '_51.png' in imageSrc:
                    pokemonIdString=str(pokemonIdCleaned)
                    #add code to add types for pokemon
                    pokeURL='https://pokeapi.co/api/v2/pokemon/'
                    pokemonMiniType5=[]
                    pokemonMiniType4=poke_special(pokemonMiniType5,pokemonIdString,pokeURL,'_51')
                    k['type'].append(pokemonMiniType4)
                elif '_52.png' in imageSrc:
                    pokemonIdString=str(pokemonIdCleaned)
                    pokeURL='https://pokeapi.co/api/v2/pokemon/'
                    pokemonMiniType5=[]
                    pokemonMiniType4=poke_special(pokemonMiniType5,pokemonIdString,pokeURL,'_52')
                    k['type'].append(pokemonMiniType4)
                else:
                    pokemonIdString=str(pokemonIdCleaned)
                    pokeURL='https://pokeapi.co/api/v2/pokemon/'
                    pokemonMiniType5=[]
                    pokemonMiniType4=poke_special(pokemonMiniType5,pokemonIdString,pokeURL,' ')
                    k['type'].append(pokemonMiniType4)"""

#pokemonJson=pokemonData.to_json()
#pokemonJsonCleaned=json.loads(pokemonJson)

#@get('/')
#def server_json():
    #return{"msg":pokemonJsonCleaned,"error":"Cannot Host Data"}

#run(host='localhost',port=8080)
#pokemon_object = json.dumps(pokemonJson,indent=4)

with open('./pokemon_data2.json',"w") as output_file:
    #output_file.write(pokemon_json.replace('\\','')+'\n')
    #output_file.write(pokemon_json_cleaned)
    #output_file.write(json.dumps({"data": pokemonJsonCleaned}, indent=4 ))
    output_file.write(json.dumps({"data": data}, indent=4 ))

#pokemonData.to_csv('./cleaned.csv')
#shutil.copy(('./pokemon_data2.json'),('./rahulvhegde.github.io/Data/pokemon_data2.json'))

In [ ]:
import requests
import json

url = "https://getpantry.cloud/apiv1/pantry/b45d3e57-17a6-498d-8aec-b8173408efb4/basket/pokemondata"

f = open('/content/pokemon_data2.json')

payload = json.load(f)
payload=json.dumps(payload)
headers = {
  'Content-Type': 'application/json'
}

response = requests.request("POST", url, headers=headers, data=payload)

print(response.text)

Your Pantry was updated with basket: pokemondata!


In [ ]:
"""from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from bs4 import BeautifulSoup
from dateutil.relativedelta import relativedelta
import pandas as pd
import requests
import re

current_year = datetime.now().year

def parse_date(date: str) -> str:
    date_format = "%A, %B %d, at %I:%M %p"
    datetime_obj = datetime.strptime(date, date_format)
    day = datetime_obj.strftime("%d")
    month = datetime_obj.strftime("%m")
    time = datetime_obj.strftime("%H:%M")
    parsed_date = f"{current_year}-{month}-{day} {time}:00"
    return parsed_date

def event_ends_next_year(start_date: str, end_date: str):
    start_month = start_date[5:7]
    end_month = end_date[5:7]
    return int(start_month) == 12 and int(end_month) < 12

def is_all_day_event(start_date: str, end_date: str):
    start_month_and_day = start_date[5:10]
    end_month_and_day = end_date[5:10]
    start_time = start_date[11:]
    end_time = end_date[11:]
    return (
        start_month_and_day == end_month_and_day
        and start_time == "00:00:00"
        and end_time == "23:59:00"
    )

def convert_to_rfc3339(date: str):
    rfc3339_format = "%Y-%m-%dT%H:%M:%S"
    date_object = datetime.strptime(date, "%Y-%m-%d %H:%M:%S")
    return date_object.strftime(rfc3339_format)

def convert_to_yyy_mm_dd(date: str):
    yyy_mm_dd_format = "%Y-%m-%d"
    date_object = datetime.strptime(date, "%Y-%m-%d %H:%M:%S")
    return date_object.strftime(yyy_mm_dd_format)

def getdata(url):
    r = requests.get(url)
    return r.text

@dataclass
class Event:
    start_time: str = field(compare=False)
    end_time: str = field(compare=False)
    summary: str
    description: str
    img_src: str
    bonus: list
    timeZone: str
    Description: str
    pokemonId: list

    def to_dict(self):
        if is_all_day_event(self.start_time, self.end_time):
            self.start_time = convert_to_yyy_mm_dd(self.start_time)
            self.end_time = convert_to_yyy_mm_dd(self.end_time)

            metadata = {
                "summary": self.summary,
                "description": self.description,
                "start": {"date": self.start_time},
                "end": {"date": self.end_time},
                "img_src": self.img_src,
                "pokemonId": self.pokemonId,
                "Bonus": self.bonus,
                "timeZone": self.timeZone,
                "Description": self.Description,
            }

        elif event_ends_next_year(self.start_time, self.end_time):
            self.start_time = convert_to_rfc3339(self.start_time)
            end_time_date_object = datetime.strptime(self.end_time, "%Y-%m-%d %H:%M:%S")
            end_time_date_object = end_time_date_object + relativedelta(year=1)
            self.end_time = end_time_date_object.strftime("%Y-%m-%d %H:%M:%S")
            self.end_time = convert_to_rfc3339(self.end_time)

            metadata = {
                "summary": self.summary,
                "description": self.description,
                "start": {"dateTime": self.start_time},
                "end": {"dateTime": self.end_time},
                "img_src": self.img_src,
                "pokemonId": self.pokemonId,
                "Bonus": self.bonus,
                "timeZone": self.timeZone,
                "Description": self.Description,
            }

        else:
            metadata = {
                "summary": self.summary,
                "description": self.description,
                "start": {"dateTime": self.start_time},
                "end": {"dateTime": self.end_time},
                "img_src": self.img_src,
                "pokemonId": self.pokemonId,
                "Bonus": self.bonus,
                "timeZone": self.timeZone,
                "Description": self.Description,
            }

        return metadata

    def __str__(self):
        return str(self.to_dict())

def main():
    events = defaultdict()
    service = webdriver.ChromeOptions()
    service.add_argument("--headless")
    service.add_argument("--no-sandbox")

    driver = webdriver.Chrome(ChromeDriverManager().install(), options=service)

    url = "https://leekduck.com/events"
    driver.get(url)
    soup = BeautifulSoup(driver.page_source, "lxml")
    soup = soup.find_all("div", class_="current-events")[0]
    soup = soup.find_all("span", class_="event-header-item-wrapper")
    timeZone = str()
    Description = str()
    event_links = set()
    img_src = list()

    for span in soup:
        event_name = span.find("a").get("href")
        if "unannounced" in event_name:
            continue
        link = f"https://leekduck.com{event_name}"
        event_links.add(link)

    for link in event_links:
        soup_bonus = []
        htmldata = getdata(link)
        soup = BeautifulSoup(htmldata, 'html.parser')
        img_src = []
        pokemonId = []

        for item in soup.find_all('img'):
            src = item.get("src")
            if src:
                src = requests.compat.urljoin(url, src)
                if len(img_src) == 0 or 'pokemon_icons' in src:
                    if '_crop' not in src:
                        if src not in img_src:
                            img_src.append(src)
                            pokemonIdData = re.findall(r'\d+', src)
                            if len(pokemonIdData) > 0 and len(pokemonIdData[0]) <= 3 and 'pokemon_icons' in src:
                                pokemonIdCleaned = str(int(pokemonIdData[0]))
                                if '_51.png' in src:
                                    pokemonId.append(str(pokemonIdCleaned + '_51'))
                                elif '_52.png' in src:
                                    pokemonId.append(str(pokemonIdCleaned + '_52'))
                                elif 'fMEGA' in src:
                                    pokemonId.append(str(pokemonIdCleaned + 'fMEGA'))
                                elif '_61.png' in src:
                                    pokemonId.append(str(pokemonIdCleaned + '_61'))
                                elif 'fHISUIAN' in src:
                                    pokemonId.append(str(pokemonIdCleaned + 'fHISUIAN'))
                                else:
                                    pokemonId.append(pokemonIdCleaned)
                elif 'shiny-icon' in src:
                    img_poke_shiny = img_src[len(img_src) - 1]
                    if '.icon.' in img_poke_shiny and (img_poke_shiny.count('.s') < 2):
                        img_poke_shiny = img_poke_shiny.split('.icon.')
                        img_poke_shiny = img_poke_shiny[0] + '.s.icon.png'
                        if img_poke_shiny not in img_src and img_poke_shiny.count('.s') < 2:
                            img_src.append(img_poke_shiny)
                            pokemonIdData = re.findall(r'\d+', img_poke_shiny)
                            if len(pokemonIdData) > 0 and len(pokemonIdData[0]) <= 3 and 'pokemon_icons' in img_poke_shiny:
                                pokemonIdCleaned = str(int(pokemonIdData[0]))
                                if '_51.png' in img_poke_shiny or '_51.s.icon.png' in img_poke_shiny:
                                    pokemonId.append(str(pokemonIdCleaned + '_51'))
                                elif '_52.png' in img_poke_shiny or '_52.s.icon.png' in img_poke_shiny:
                                    pokemonId.append(str(pokemonIdCleaned + '_52'))
                                elif 'fMEGA' in img_poke_shiny:
                                    pokemonId.append(str(pokemonIdCleaned + 'fMEGA'))
                                elif '_61.png' in img_poke_shiny or '_61.s.icon.png' in img_poke_shiny:
                                    pokemonId.append(str(pokemonIdCleaned + '_61'))
                                elif 'fHISUIAN' in img_poke_shiny:
                                    pokemonId.append(str(pokemonIdCleaned + 'fHISUIAN'))
                                else:
                                    pokemonId.append(pokemonIdCleaned)
                    elif '_shiny' in img_poke_shiny and (img_poke_shiny.count('_shiny') < 2 and img_poke_shiny.count('.s') < 2):
                        img_poke_shiny = img_poke_shiny.split('.png')[0] + '_shiny.png'
                        if img_poke_shiny not in img_src and img_poke_shiny.count('_shiny') < 2 and img_poke_shiny.count('.s') < 2:
                            img_src.append(img_poke_shiny)
                            pokemonIdData = re.findall(r'\d+', img_poke_shiny)
                            if len(pokemonIdData) > 0 and len(pokemonIdData[0]) <= 3 and 'pokemon_icons' in img_poke_shiny:
                                pokemonIdCleaned = str(int(pokemonIdData[0]))
                                if '_51.png' in img_poke_shiny or '_51_shiny.png' in img_poke_shiny:
                                    pokemonId.append(str(pokemonIdCleaned + '_51'))
                                elif '_52.png' in img_poke_shiny or '_52_shiny.png' in img_poke_shiny:
                                    pokemonId.append(str(pokemonIdCleaned + '_52'))
                                # Handle other cases here if needed

        for soups in soup.find_all("div", class_="bonus-text"):
            soup_bonus.append(soups.string)

        for span in soup.find_all('span', {'id': 'event-time-end'}):
            timeZoneString = span.string
            timeZone = 'Local Time' if len(timeZoneString.split('M')) == 1 else timeZoneString.split('M')[1]

        driver.get(link)
        soup = BeautifulSoup(driver.page_source, "lxml")
        for p in soup.find_all("div", class_="event-description"):
            data = p.find_all('p')
            desc = str(data[0]).replace('<p>', '')
            Description = desc.replace('</p>', '')

        title = soup.find("h1").text.strip()
        start_date = (
            WebDriverWait(driver, 10)
            .until(EC.presence_of_element_located((By.ID, "event-date-start")))
            .text.strip()
            .rstrip(",")
            .replace("  ", " ")
        )

        start_time = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "event-time-start"))
        ).text.split("M")[0] + "M".replace("  ", " ")

        complete_start_date = f"{start_date}, {start_time}"

        end_date = (
            WebDriverWait(driver, 10)
            .until(EC.presence_of_element_located((By.ID, "event-date-end")))
            .text.strip()
            .rstrip(",")
            .replace("  ", " ")
        )

        end_time = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "event-time-end"))
        ).text.split("M")[0] + "M".replace("  ", " ")

        complete_end_date = f"{end_date}, {end_time}"

        if start_date != "None":
            parsed_start_date = parse_date(complete_start_date)
            parsed_end_date = parse_date(complete_end_date)

            new_event = Event(parsed_start_date, parsed_end_date, title, link, img_src, soup_bonus, timeZone, Description, pokemonId)
            events[link] = new_event

    df = pd.DataFrame(events, index=[0]).T
    df.columns = ['Summary']
    df.to_csv('./file.csv')
    driver.quit()


if __name__ == "__main__":
    main()
"""

'from selenium import webdriver\nfrom webdriver_manager.chrome import ChromeDriverManager\nfrom collections import defaultdict\nfrom dataclasses import dataclass, field\nfrom datetime import datetime\nfrom bs4 import BeautifulSoup\nfrom dateutil.relativedelta import relativedelta\nimport pandas as pd\nimport requests\nimport re\n\ncurrent_year = datetime.now().year\n\ndef parse_date(date: str) -> str:\n    date_format = "%A, %B %d, at %I:%M %p"\n    datetime_obj = datetime.strptime(date, date_format)\n    day = datetime_obj.strftime("%d")\n    month = datetime_obj.strftime("%m")\n    time = datetime_obj.strftime("%H:%M")\n    parsed_date = f"{current_year}-{month}-{day} {time}:00"\n    return parsed_date\n\ndef event_ends_next_year(start_date: str, end_date: str):\n    start_month = start_date[5:7]\n    end_month = end_date[5:7]\n    return int(start_month) == 12 and int(end_month) < 12\n\ndef is_all_day_event(start_date: str, end_date: str):\n    start_month_and_day = start_date[5:10]

In [ ]:
from urllib.request import Request,urlopen

import base64

linkCleaned = 'https://leekduck.com/assets/img/pokemon_icons/pm921.icon.png'

request_site = Request(linkCleaned, headers={"User-Agent": "Mozilla/5.0"})
encodedData = base64.b64encode(urlopen(request_site).read())
print(encodedData)

imgdata = base64.b64decode(encodedData)
filename = 'some_image.jpg'
with open(filename, 'wb') as f:
    f.write(imgdata)

b'iVBORw0KGgoAAAANSUhEUgAAAQAAAAEACAYAAABccqhmAAAf2ElEQVR42u3df5BdZZ3n8fdz7o/+lXS6Q5JuSSICASIiYIk/wFJxIDBolSIatnBHjKuO1oyjziyWs+5Wgbs7LpbMTA0rOzNb7jAOKg6sLLAlMkR3WGumCL8WAsiPIQQwIaQ7Pzrp7vvznOd59o/nnHvPvX27k/AzJJ8Xdeh7z/2Ze87zfb7Pc57zHBARERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERERETk6Gf0ER8w28/pp5FBF+glEFABERAFARBQAREQBQN4w1AEoIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiIiRyfvx9bpV5DDja4M9GoX/ArjfhNXKQjI4cjoJ3gVC//jbOAXXA9AI/fAFWMXGDOx8Y2TvaxZY8yWLdqiCgByKAXn2tw1+xo9nnCYBgLv16xha+V8npj+BNPJSi5b/VEFAAUAOZRCtJHreYINCwYAwHz98NoGftPSv+Cx/ZdRtcs7HljLI+YCztCWVR+AHIx84V/AtvuHvN+++OLDovBvZif37f3KnMKfPX4Xm7VhlQHIgunzujEArtm4M78+rheJDEQGTO5X94uTsCG+vHa4Y8OYJ2cO7XPH1r2c5kSruTJTaq2zDpo23J6ejQEYu4Lx1hPG/DpjzA+11ZUBSKvgbpxgy+RF3eubtr3EFp